<div align="center">

# Buradan başlayın: Bilgisayar, Jupyter ve ilk Python satırları

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Eğitimin ana sayfası](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/README.tr.md) · [Kurulum](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/SETUP.tr.md) · [1. hafta](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/tree/main/tr/hafta-01) · [Colab'da aç (yedek yol)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/baslangic/NB00_baslangic.ipynb)

</div>

Bu Jupyter defteri bilgisayarı yaklaşık 30 dakikada eğitime hazırlar. Python paketlerini denetler, ders klasörünü bulur, Jupyter'i ve klinik sayılarla ilk Python satırlarını tanıtır ve ilk elektrokardiyogramı (EKG) çizer. Son olarak bir yapay zekâ (YZ) asistanına ilk prompt (istem) gönderilir ve dönen kod denetlenir. Her şey çalışıyorsa son hücre **1. haftaya hazırsınız** yazar.

**Nasıl kullanılır?** Her gri kutu bir kod hücresidir. Bir hücrenin içine tıklayın ve onu çalıştırmak için Shift+Enter tuşlarına basın; hücreleri yukarıdan aşağıya sırayla çalıştırın. Çalışmakta olan bir hücrenin solunda `[*]`, çalışması biten bir hücrenin solunda bir sayı görünür. *Run → Run All Cells* bütün defteri bir kerede çalıştırır.

## 1. Paketler kurulu mu?

Aşağıdaki hücre 1. haftanın paketlerini tek tek içe aktarır ve sürümlerini yazar. NumPy, pandas, SciPy ve Matplotlib zorunludur ve Anaconda ile birlikte gelir. Diğer üçü isteğe bağlıdır: Eğitimin defterleri bu paketler kuruluysa onları, kurulu değilse yerleşik bir yedek yolu kullanır.

In [ ]:
import importlib
import platform

print("Python", platform.python_version())
REQUIRED = ["numpy", "pandas", "scipy", "matplotlib"]
OPTIONAL = ["ipywidgets", "wfdb", "neurokit2"]
missing = []
for name in REQUIRED + OPTIONAL:
    try:
        module = importlib.import_module(name)
        print(f"{name:12s} {getattr(module, '__version__', '')}")
    except ImportError:
        if name in REQUIRED:
            missing.append(name)
            print(f"{name:12s} EKSİK: Anaconda'yı yeniden kurun ya da şunu çalıştırın: pip install {name}")
        else:
            print(f"{name:12s} kurulu değil (isteğe bağlı)")

## 2. Ders klasörü nerede?

Eğitimin defterleri `vclinic` ders paketini ve `data` klasöründeki veri dosyalarını kullanır. Aşağıdaki hücre ders klasörünü bu defterin bulunduğu klasörden başlayarak arar. Defter tek başına açıldıysa, örneğin Google Colab'da, paketi GitHub'dan indirir.

In [ ]:
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopya, örneğin Colab'da: Paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    for rel in ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]:
        target = ROOT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders paketi şu klasöre indirildi:", ROOT)
else:
    print("Ders klasörü:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 3. Beş dakikada Jupyter

Bir Jupyter defteri, bunun gibi metin hücrelerini kod hücreleriyle bir arada tutar. Python bir hücrenin oluşturduğu her şeyi bellekte saklar; bu yüzden hücrelerin çalışma sırası önemlidir: Bir hücre bir adı ancak o adı oluşturan hücre çalıştıktan sonra kullanabilir. Defter karışırsa, örneğin hücreler sırasız çalıştırıldıktan sonra, *Kernel → Restart Kernel and Run All Cells* komutu defteri temiz bir durumdan yeniden başlatır.

Üç alışkanlık ilk günden işe yarar. Tab tuşu adları tamamlar; örneğin `pri` yazıp Tab'a basınca `print` önerilir. Bir adın ardından gelen soru işareti yardımını gösterir, örneğin `print?`. Ctrl+S (macOS'ta Cmd+S) ile kaydetmek, defteri sonuçlarıyla birlikte saklar.

## 4. İlk Python satırları

Bir değişken, bir değerin adıdır. Aşağıdaki hücre bir hastanın ağırlığını ve boyunu saklar ve beden kitle indeksini, $\text{BKİ} = \text{ağırlık} / \text{boy}^2$, kg/m² cinsinden hesaplar. `**` işleci üs alır, `round(x, 1)` ise bir ondalığa yuvarlar.

In [ ]:
weight_kg = 82
height_m = 1.76
bmi = weight_kg / height_m ** 2
print("BKİ:", round(bmi, 1), "kg/m²")

Aşağıdaki küçük fonksiyon iki alıştırmanın cevaplarını denetler. Bir cevap beklenen değere yeterince yakınsa *doğru* yazar.

In [ ]:
def check(name, answer, expected, tolerance):
    if answer is None:
        print(f"{name}: önce cevabınızı yazın.")
    elif abs(answer - expected) <= tolerance:
        print(f"{name}: doğru.")
    else:
        print(f"{name}: henüz değil. Formülü ve birimleri denetleyin.")

**Alıştırma 0.1.** Ortalama arter basıncı çoğu zaman $\text{OAB} = \text{DKB} + (\text{SKB} - \text{DKB})/3$ ile kestirilir. Bunu 121/71 mmHg'lik bir kan basıncı için hesaplayın: `None` yerine `sbp` ve `dbp` değişkenlerini kullanan formülü yazın.

In [ ]:
sbp = 121
dbp = 71
mean_arterial_pressure = None  # None yerine formülü yazın
check("Alıştırma 0.1", mean_arterial_pressure, 87.67, 0.05)

**Alıştırma 0.2.** Dakikadaki atım sayısı (bpm) cinsinden kalp hızı, 60'ın saniye cinsinden RR aralığına bölümüdür. 706 ms'lik bir RR aralığı için kalp hızını hesaplayın. Birime dikkat edin: 706 ms, 0,706 s'dir.

In [ ]:
rr_ms = 706
heart_rate_bpm = None  # None yerine formülü yazın
check("Alıştırma 0.2", heart_rate_bpm, 84.99, 0.05)

In [ ]:
#@title Çözümler (denedikten sonra açın)
mean_arterial_pressure = dbp + (sbp - dbp) / 3
heart_rate_bpm = 60 / (rr_ms / 1000)
print(round(mean_arterial_pressure, 2), "mmHg;", round(heart_rate_bpm, 2), "bpm")

## 5. Bir EKG'ye ilk bakış

`vclinic` ders paketi sanal kliniğin hastalarını oluşturur. Aşağıdaki hücre 500 sentetik hastadan oluşan kaydı oluşturur, 1. haftanın vakası olan VC-0317 numaralı hastayı alır ve D2 derivasyonunun on saniyesini EKG kâğıdı üzerinde çizer. 1. hafta bu resmin her adımını açıklar.

In [ ]:
import matplotlib.pyplot as plt

import vclinic
from vclinic import teach

teach.set_language("tr")
clinic = vclinic.make_cohort(500)
case = vclinic.patient_ecg(clinic, "VC-0317")
print(case)

fig, ax = plt.subplots(figsize=(12, 3))
teach.plot_strip(ax, case.time, case.lead("II"), label="D2")
plt.show()

## 6. İlk promptunuz

Eğitim, 1. haftadan itibaren kod yazmak için YZ asistanlarını tek bir katı kuralla kullanır: Prompt veriyi tarif eder, hastayı asla tarif etmez. Eğitimdeki her alıştırma [prompt yolundan](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.tr.md) da çözülebilir: Bir prompt asistana gönderilir, dönen kod çalıştırılır ve denetlenir. Her prompt, yedi alanı olan eğitimin prompt kartıyla yazılır.

Aşağıdaki promptu bir YZ asistanına kopyalayın; örneğin JupyterLab'deki Anaconda Assistant'a ya da Colab'daki Gemini'ye. Asistanın döndürdüğü kodu sonraki hücreye yapıştırın ve çalıştırın. Ondan sonraki hücre fonksiyonu promptta geçmeyen bir durumla denetler. Asistan kullanmıyorsanız çözüm hücresini açın ve satır satır okuyun: 1. haftanın Python adımı 1 fonksiyonları açıklamaktadır.

<pre>
Rol: Python öğrenen bir kardiyoloğa yardım ediyorsun.
Bağlam: Python 3 kullanan bir Jupyter defteri; NumPy np adıyla içe aktarılabilir.
Veri: Bir EKG derivasyonundaki R tepelerinin konumları (örnek indeksi olarak) bir listede; örnekleme
frekansı fs, Hz cinsinden.
Görev: Ortalama kalp hızını dakikadaki atım sayısı olarak döndüren bir mean_heart_rate(peaks, fs)
fonksiyonu yaz. Saniye cinsinden RR aralıkları, ardışık tepe konumları arasındaki farkların fs'ye
bölümüdür; kalp hızı ise 60'ın bu aralıkların ortalamasına bölümüdür.
Kısıtlar: Yalnızca NumPy kullan. Hiçbir dosya okuma, hiçbir paket kurma.
Çıktı: Yalnızca kodu, tek bir blok olarak ver; her adımın üstüne kısa bir yorum satırı ekle.
Test: Bir assert ifadesiyle denetle: 360 Hz'de 0, 360 ve 720 konumlarındaki tepeler 60 bpm vermeli.
</pre>

In [ ]:
# YZ asistanının kodunu buraya yapıştırın ve çalıştırın.

In [ ]:
# İlk promptun, promptta geçmeyen bir durumla denetimi
try:
    answer = mean_heart_rate([0, 300, 600, 900], 500)
except NameError:
    answer = None
check("İlk prompt", answer, 100.0, 0.05)

In [ ]:
#@title Çözüm (denedikten sonra açın)
import numpy as np


def mean_heart_rate(peaks, fs):
    # saniye cinsinden RR aralıkları
    rr_s = np.diff(peaks) / fs
    # dakikadaki atım sayısı olarak ortalama kalp hızı
    return 60 / rr_s.mean()


assert round(mean_heart_rate([0, 360, 720], 360), 1) == 60.0
print(round(mean_heart_rate([0, 300, 600, 900], 500), 1), "bpm")

**Açıkla ve kaydet.** Asistandan kodunu satır satır açıklamasını isteyin, ardından kodu bir kez de kendi sözcüklerinizle açıklayın. Denetim, 500 Hz'de her 300 örnekte bir gelen tepeleri kullandı: RR aralığı 300 / 500 = 0,6 s, kalp hızı da 60 / 0,6 = 100 bpm'dir. Bir kod, sonucu böyle bir denetimden geçtiğinde ve her satırı açıklanabildiğinde kabul edilir. 1. hafta kod güvenlik kontrol listesini ve bir asistanın her kullanımının kaydedildiği YZ kullanım günlüğünü ekler. Bir prompt hiçbir zaman gerçek hastaların adlarını, doğum tarihlerini, protokol numaralarını ya da kayıtlarını içermez.

## 7. 1. haftaya hazır mısınız?

Son hücre bu defterdeki denetimleri bir araya getirir.

In [ ]:
data_ok = (ROOT / "vclinic" / "__init__.py").exists()
if missing:
    print("Önce eksik paketleri kurun:", ", ".join(missing))
elif not data_ok:
    print("Ders klasörü bulunamadı. SETUP.tr.md dosyasına bakın.")
else:
    print("vclinic", vclinic.__version__, "·", "ders klasörü", ROOT)
    print("1. haftaya hazırsınız.")

**Sırada:** [1. hafta sayfasını](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/tree/main/tr/hafta-01) ve `tr/hafta-01/NB01_python_ve_ekg_temelleri.ipynb` Jupyter defterini açın.

---

<div align="center">

**Prof. Dr. Utku Köse**

Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*Bu eğitim, alandaki güncel gelişmeler doğrultusunda güncellenmektedir. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>